# Project: Cardinal Store - Forecasting Retail Demand
## Orientation and Setup


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import sklearn
from sklearn.preprocessing import StandardScaler, LabelEncoder

from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score

from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, mean_squared_error, r2_score, mean_pinball_loss, mean_absolute_error)

import lightgbm as lgb
from lightgbm import LGBMClassifier, LGBMRegressor

from prophet import Prophet

In [2]:
# pip install prophet

In [3]:
df_fact_sales = pd.read_csv(r'C:\Users\USER\Downloads\Amdari\Projects\Cardinal Stores - Forecasting Retail Demand\cardinal_stores_data\fact_sales.csv')
df_fact_inventory = pd.read_csv(r'C:\Users\USER\Downloads\Amdari\Projects\Cardinal Stores - Forecasting Retail Demand\cardinal_stores_data\fact_inventory.csv')
df_fact_replenishment = pd.read_csv(r'C:\Users\USER\Downloads\Amdari\Projects\Cardinal Stores - Forecasting Retail Demand\cardinal_stores_data\fact_replenishment.csv')
df_dim_store = pd.read_csv(r'C:\Users\USER\Downloads\Amdari\Projects\Cardinal Stores - Forecasting Retail Demand\cardinal_stores_data\dim_store.csv')
df_dim_product = pd.read_csv(r'C:\Users\USER\Downloads\Amdari\Projects\Cardinal Stores - Forecasting Retail Demand\cardinal_stores_data\dim_product.csv')
df_dim_promotion = pd.read_csv(r'C:\Users\USER\Downloads\Amdari\Projects\Cardinal Stores - Forecasting Retail Demand\cardinal_stores_data\dim_promotion.csv')
df_dim_calendar = pd.read_csv(r'C:\Users\USER\Downloads\Amdari\Projects\Cardinal Stores - Forecasting Retail Demand\cardinal_stores_data\dim_calendar.csv')
df_dim_external_factors = pd.read_csv(r'C:\Users\USER\Downloads\Amdari\Projects\Cardinal Stores - Forecasting Retail Demand\cardinal_stores_data\dim_external_factors.csv')


In [4]:
import json

with open(r'C:\Users\USER\Downloads\Amdari\Projects\Cardinal Stores - Forecasting Retail Demand\cardinal_stores_data\validation_report.json', 'r') as f:
    vld_rpt = json.load(f)

print(type(vld_rpt))

<class 'dict'>


In [5]:
print(vld_rpt.keys())

dict_keys(['passed', 'failed', 'warnings', 'score', 'dataset_ready', 'row_counts'])


In [6]:
import pprint
pprint.pprint(vld_rpt)

{'dataset_ready': True,
 'failed': [],
 'passed': ['dim_product: no duplicate rows',
            'dim_product: missing values acceptable',
            'dim_store: no duplicate rows',
            'dim_store: missing values acceptable',
            'fact_sales: no duplicate rows',
            'fact_sales: missing values acceptable',
            'fact_inventory: no duplicate rows',
            'fact_inventory: missing values acceptable',
            'fact_replenishment: no duplicate rows',
            'fact_replenishment: missing values acceptable',
            'dim_promotion: no duplicate rows',
            'dim_promotion: missing values acceptable',
            'fact_sales.units_sold >= 0',
            'fact_sales.gross_revenue >= 0',
            'fact_inventory.on_hand_units >= 0',
            'fact_replenishment.lead_time_days > 0',
            'fact_sales.sku_id ⊆ dim_product',
            'fact_sales.store_id ⊆ dim_store',
            'fact_inventory.sku_id ⊆ dim_product',
         

In [7]:
dfs = {
    "df_fact_sales": df_fact_sales,
    "df_fact_inventory": df_fact_inventory,
    "df_fact_replenishment": df_fact_replenishment,
    "df_dim_store": df_dim_store,
    "df_dim_product": df_dim_product,
    "df_dim_promotion": df_dim_promotion,
    "df_dim_calendar": df_dim_calendar,
    "df_dim_external_factors": df_dim_external_factors
}

for name, df in dfs.items():
    print(f"{name}: shape = {df.shape}")

df_fact_sales: shape = (356596, 11)
df_fact_inventory: shape = (438600, 9)
df_fact_replenishment: shape = (30107, 8)
df_dim_store: shape = (12, 7)
df_dim_product: shape = (50, 9)
df_dim_promotion: shape = (230, 6)
df_dim_calendar: shape = (731, 6)
df_dim_external_factors: shape = (8772, 5)


In [8]:
for name, df in dfs.items():
    print(f"\n{name} dtypes:")
    print(df.dtypes)


df_fact_sales dtypes:
sales_id             int64
sku_id               int64
store_id             int64
channel             object
sales_date          object
units_sold           int64
unit_price         float64
discount_amount    float64
gross_revenue      float64
was_on_promo          bool
lost_sales_flag       bool
dtype: object

df_fact_inventory dtypes:
inventory_id       int64
sku_id             int64
store_id           int64
snapshot_date     object
on_hand_units      int64
on_order_units     int64
reorder_point      int64
safety_stock       int64
stockout_flag       bool
dtype: object

df_fact_replenishment dtypes:
po_id                   object
sku_id                   int64
store_id                 int64
supplier_id              int64
order_date              object
actual_delivery_date    object
order_quantity           int64
lead_time_days           int64
dtype: object

df_dim_store dtypes:
store_id        int64
store_type     object
region         object
state          obje